# JAX adata EDA

Basic exploratory analysis for the downloaded JAX AnnData files: file inventory, metadata summaries, embryonic staging exploration, and a full-data UMAP.

In [ ]:
from pathlib import Path
import os
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from jax_adata_streaming import StreamingUmapConfig, run_streaming_umap, summarize_cell_metadata

DATA_DIR = Path(os.environ.get("ADATA_DIR", PROJECT_ROOT / "downloads" / "adata")).resolve()
OUT_DIR = Path(os.environ.get("EDA_OUT_DIR", PROJECT_ROOT / "outputs" / "jax_adata_eda")).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    "adata_JAX_dataset_1.h5ad",
    "adata_JAX_dataset_2.h5ad",
    "adata_JAX_dataset_3.h5ad",
    "adata_JAX_dataset_4.h5ad",
    "df_cell.csv",
    "df_gene.csv",
]
H5AD_FILES = [DATA_DIR / f"adata_JAX_dataset_{i}.h5ad" for i in range(1, 5)]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

## File Inventory

In [ ]:
inventory = []
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    inventory.append({
        "file": name,
        "exists": path.exists(),
        "size_gb": path.stat().st_size / 1024**3 if path.exists() else np.nan,
    })
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT_DIR / "file_inventory.csv", index=False)
display(inventory)

missing = inventory.loc[~inventory["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError(f"Missing expected files: {missing}")

## AnnData Structure and Metadata

In [ ]:
def close_backed(adata):
    if getattr(adata, "isbacked", False) and getattr(adata, "file", None) is not None:
        adata.file.close()

summaries = []
obs_frames = []
obsm_keys_by_file = {}
var_names_by_file = {}

for path in H5AD_FILES:
    a = sc.read_h5ad(path, backed="r")
    summaries.append({
        "file": path.name,
        "n_obs": a.n_obs,
        "n_vars": a.n_vars,
        "n_obs_columns": len(a.obs.columns),
        "n_var_columns": len(a.var.columns),
        "obsm_keys": ",".join(a.obsm.keys()),
        "layers": ",".join(a.layers.keys()),
    })
    obs = a.obs.copy()
    obs["source_file"] = path.name
    obs_frames.append(obs)
    obsm_keys_by_file[path.name] = list(a.obsm.keys())
    var_names_by_file[path.name] = pd.Index(a.var_names.astype(str))
    close_backed(a)

adata_summary = pd.DataFrame(summaries)
obs_all = pd.concat(obs_frames, axis=0, join="outer", copy=False)
obs_all.to_csv(OUT_DIR / "obs_metadata_all.csv")
adata_summary.to_csv(OUT_DIR / "adata_summary.csv", index=False)

display(adata_summary)
print(f"Combined metadata rows: {obs_all.shape[0]:,}")
print(f"Combined metadata columns: {obs_all.shape[1]:,}")

In [ ]:
common_genes = None
for genes in var_names_by_file.values():
    common_genes = genes if common_genes is None else common_genes.intersection(genes)

gene_overlap = pd.DataFrame({
    "metric": ["common_genes_across_h5ad"],
    "value": [len(common_genes)],
})
gene_overlap.to_csv(OUT_DIR / "gene_overlap.csv", index=False)
display(gene_overlap)

In [ ]:
def example_values(s):
    vals = s.dropna().astype(str).unique()[:5]
    return "; ".join(vals)

metadata_summary = pd.DataFrame({
    "column": obs_all.columns,
    "dtype": [str(obs_all[c].dtype) for c in obs_all.columns],
    "missing_fraction": [float(obs_all[c].isna().mean()) for c in obs_all.columns],
    "n_unique": [int(obs_all[c].nunique(dropna=True)) for c in obs_all.columns],
    "examples": [example_values(obs_all[c]) for c in obs_all.columns],
})
metadata_summary.sort_values(["missing_fraction", "n_unique"], ascending=[True, False]).to_csv(
    OUT_DIR / "metadata_column_summary.csv", index=False
)
display(metadata_summary.head(30))

## Metadata Plots

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR / name, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=adata_summary, x="file", y="n_obs", color="#4c78a8")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Cells")
plt.xlabel("")
plt.title("Cells per AnnData file")
savefig("cells_per_file.png")

completeness = metadata_summary.sort_values("missing_fraction").head(40).copy()
completeness["present_fraction"] = 1 - completeness["missing_fraction"]
plt.figure(figsize=(8, max(4, 0.18 * len(completeness))))
sns.barplot(data=completeness, y="column", x="present_fraction", color="#59a14f")
plt.xlabel("Present fraction")
plt.ylabel("")
plt.title("Most complete metadata columns")
savefig("metadata_completeness_top40.png")

## Embryonic Staging and Time Metadata

In [ ]:
stage_regex = re.compile(r"stage|embry|gest|age|time|dpc|\be\d|day", re.IGNORECASE)
stage_cols = [c for c in obs_all.columns if stage_regex.search(str(c))]

stage_summary = metadata_summary[metadata_summary["column"].isin(stage_cols)].sort_values("column")
stage_summary.to_csv(OUT_DIR / "staging_metadata_summary.csv", index=False)
display(stage_summary)

def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")[:80]

for col in stage_cols[:12]:
    s = obs_all[col]
    if pd.api.types.is_numeric_dtype(s):
        plt.figure(figsize=(7, 4))
        sns.histplot(s.dropna(), bins=50, color="#f28e2b")
        plt.title(col)
        plt.xlabel(col)
        savefig(f"stage_hist_{safe_name(col)}.png")
    else:
        counts = s.astype(str).replace("nan", np.nan).dropna().value_counts().head(40).reset_index()
        counts.columns = [col, "n_cells"]
        counts.to_csv(OUT_DIR / f"stage_counts_{safe_name(col)}.csv", index=False)
        plt.figure(figsize=(8, max(4, 0.22 * len(counts))))
        sns.barplot(data=counts, y=col, x="n_cells", color="#f28e2b")
        plt.title(col)
        plt.xlabel("Cells")
        plt.ylabel("")
        savefig(f"stage_counts_{safe_name(col)}.png")

## Cell Labels and Per-Embryo Staging

In [ ]:
cell_metadata = summarize_cell_metadata(DATA_DIR / "df_cell.csv", OUT_DIR)
cell_metadata_presence = cell_metadata["presence"]
cell_label_counts = cell_metadata["label_summary"]
embryo_staging = cell_metadata["embryo_staging"]

display(cell_metadata_presence)
display(cell_label_counts.groupby("column", group_keys=False).head(10))
display(embryo_staging.head(20))

## Full-Data UMAP

In [ ]:
streaming_config = StreamingUmapConfig.from_env()
umap_df, streaming_summary = run_streaming_umap(
    h5ad_files=H5AD_FILES,
    out_dir=OUT_DIR,
    adata_summary=adata_summary,
    common_genes=common_genes,
    obs_all=obs_all,
    stage_cols=stage_cols,
    savefig=savefig,
    cell_metadata_path=DATA_DIR / "df_cell.csv",
    config=streaming_config,
)

streaming_summary = pd.DataFrame([streaming_summary])
streaming_summary.to_csv(OUT_DIR / "streaming_umap_summary.csv", index=False)
display(streaming_summary)
print(f"Detailed streaming progress: {OUT_DIR / 'streaming_umap_progress.log'}")

In [ ]:
candidate_color_cols = ["source_file"] + stage_cols[:4]
candidate_color_cols = [c for c in candidate_color_cols if c in obs_all.columns or c == "source_file"]

if "umap_df" in globals():
    plot_df = umap_df.copy()
    for col in stage_cols[:4]:
        if col in obs_all.columns and col not in plot_df.columns:
            aligned = obs_all[[col]].copy()
            aligned.index = aligned.index.astype(str)
            plot_df = plot_df.join(aligned, how="left")
    for col in [c for c in candidate_color_cols if c in plot_df.columns and c != "source_file"]:
        top = plot_df[col].astype(str).value_counts().head(20).index
        tmp = plot_df[plot_df[col].astype(str).isin(top)].copy()
        plt.figure(figsize=(7, 6))
        sns.scatterplot(data=tmp, x="UMAP1", y="UMAP2", hue=col, s=1, linewidth=0, alpha=0.35)
        plt.legend(markerscale=6, bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.title(f"Full-data UMAP by {col}")
        savefig(f"full_umap_by_{safe_name(col)}.png")

print("EDA complete.")

## UMAP by Cell Type

In [ ]:
from IPython.display import Image, display

for figure in [
    OUT_DIR / "full_umap_by_major_trajectory.png",
    OUT_DIR / "full_umap_by_celltype_update_top30.png",
]:
    display(Image(filename=figure))